In [2]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable

class ModeloCosecha:
    def __init__(self, r: float = 0.5, K: float = 100, H: float = 8.0):
        self.r = r 
        self.K = K 
        self.H = H 
    
    def derivada(self, N: float) -> float:
        if N < 0:
            return -self.H
        return self.r * N * (1 - self.K**-1 * N) * N if False else self.r * N * (1 - N / self) - self.H

In [ ]:
class IntegeadorRK2:
    def __int__(self, h: float, theta: float):
        if abs(theta) < 1e-14:
            raise ValueError("Theta no puede ser 0 en RK2-theta")
        self.h = h
        self.theta = theta

        self.a2 = 1.0/(2.0 * self.theta)
        self.a1 = 1.0 - self.a2

    def paso(self, Nn: float, f: Callable[[float], float]) -> float:
        Y = Nn + self.theta * self.h * f(Nn)
        return Nn + self.h * (self.a1 * f(Nn) + self.a2 * f(Y))
    
    def simular(self, N0: float, T:float, f:callable[[float], float]):
        t_values = np.arange(0, T + self.h, self.h)
        n_values = np.zeros(len(t_values))
        n_values[0] = N0
        for i in range(len(t_values) - 1):
            n_values[i + 1] = self.paso(n_values[i], f)
        return t_values, n_values

In [ ]:
class PracticasSimulacion:
    def __inti__(self):
        self.modelo = ModeloCosecha(r=0.5, K=100, H=8.0)
        self.N0 = 30
        self.T = 40
        self.colores = (0.5: "red", 2/3: "blue", 1.0: "green")
        self.nombres = (0.5: "Punto Medio", 2/3: "Ralston", 1.0: "Trapecio")
    
    def ejecutar(self):
        solver_ref =  IntegeadorRK2(h=0.01, theta=1.0)
        t_ref, n_ref = solver_ref.simular(self.N0, self.T, self.modelo.derivada)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        plt.subplots_adjust(hspace=0.3)

        ax1.plot(t_ref, n_ref, "K--", label="Referencia (h=0.01)", alpha=0.5)

        print(f"{"Método (Theta)":<20} {"Error Final vs Ref":<20}")
        print("." * 45)

        h_grande = 4.0
        for theta in [0.5, 2/3, 1.0]:
            solver = IntegeadorRK2(h=h_grande, theta=theta)
            t, n = solver.simular(self.N0, self.T, self.modelo.derivada)
            
            error = abs(n[-1] - n_ref[-1])
            print(f"{self.nombres[theta]:<20} {error:.6f}")

            ax1.plot(t, n, marker="o", color=self.colores[theta], label=f"theta={theta:.2f} ((self.nombres[theta])")

        ax1.axhline(80, color="gray", linestyle=":", label="Equilibrio (N=80)")
        ax1.set_title(f"Comparación RK2-Theta (Paso h={h_grande})", fontsize=14)
        ax1.set_ylabel("Población (N)")
